# Canary-1B-v2 with and without decoder biasing

Run on JupyterHub with a GPU and the kernel "NeMo ASR" (`~/nemoenv`, see `../environments/README.md`). If Jupyter asks for a kernel, choose "NeMo ASR".

Before the first run, apply the NeMo fix once in the terminal: `~/nemoenv/bin/python <repository>/environments/patch_nemo_py310.py`. Without it, Canary fails on clips longer than 40 s and cell 4 stores them as errors.

The notebook decodes every segment of `manifest_biasing_full.json` twice with `nvidia/canary-1b-v2` at beam size 4. The two arms differ only in the boosting:

| Arm | Boosting |
|---|---|
| `beam4_alpha0.0` | none |
| `beam4_alpha1.0` | weight 1.0 toward the term list of the segment's meeting (names and terms from its previous protocol) |

The arms and their weights come from `config.arms` in the manifest. Apart from the working folder, the code cells are the code that produced the thesis decodes.

**Reads** (relative to this folder, `recognizers/`, where the notebook runs):
- `manifest_biasing_full.json`: the segments, the arms and the term list of each meeting.
- `goldset/clips_v10/*.mp3`: the clips, unpacked from `clips_v10_gold.zip` and `clips_v10_rest.zip`.

**Writes:**
- `bias_full/sweep_full.json`: per arm and segment, the `meeting_id` and the `text`, or an `error` if the decode failed.
- `bias_full_results.zip`: the folder `bias_full/` as one archive.

**Run the cells in order:**
1. Checks the working directory, the GPU, the manifest and that every clip is present, and creates `bias_full/`.
2. Loads Canary and the decoding helpers.
3. Smoke test on up to three gold clips of one meeting, in both arms. The output must be German text.
4. Full run: gold segments first, then the rest, one meeting at a time in both arms. Saves every 50 decodes. At the end it prints, per arm, how many segments were decoded and how many failed.
5. Counts the segments whose text the boosting changed and zips `bias_full/`. If the count is 0, the boosting had no effect: do not score the run.

**After a crash:** re-run cells 1 and 2, then cell 4. Cell 4 skips the segments that already have a text and decodes the failed ones again.

**Next:** copy `bias_full/sweep_full.json` to `../evaluation/bias_full/` and the manifest to `../evaluation/dist/cluster_biasing_full/`, then score it with `../evaluation/score_biasing_full.py` (see `README.md`).

In [ ]:
# 1. Environment check: working directory, GPU, manifest and clips.
import sys, os, json, time, gc, copy, tempfile
from pathlib import Path
import torch

WORK = Path.cwd()
assert WORK.exists(), f"{WORK} missing - upload the package there first"
os.chdir(WORK); sys.path.insert(0, str(WORK))
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
assert torch.cuda.is_available(), "no GPU visible - check the kernel and the node"
DEVICE = "cuda"

MAN   = json.loads(Path("manifest_biasing_full.json").read_text(encoding="utf-8"))
SEGS  = MAN["segments"]
TERMS = MAN["bias_terms"]
CLIPS = Path("goldset/clips_v10")
assert CLIPS.exists(), ("goldset/clips_v10 missing - unzip clips_v10_gold.zip AND clips_v10_rest.zip "
                        "(from dist/cluster_v10) in this folder")
present = {p.name for p in CLIPS.glob("*.mp3")}
missing = [s["clip"] for s in SEGS if s["clip"] not in present]
assert not missing, f"{len(missing)} clips missing, e.g. {missing[:3]} - unzip both clip zips"
OUT = Path("bias_full"); OUT.mkdir(exist_ok=True)
print(f"{len(SEGS)} clips ready ({sum(s['gold'] for s in SEGS)} gold), {len(TERMS)} meetings")
print("environment OK")

In [ ]:
# 2. Load Canary-1B-v2 and the decoding helpers.
import numpy as np, librosa, soundfile as sf
from omegaconf import OmegaConf
from nemo.collections.asr.models import ASRModel

SR = 16000
canary = ASRModel.from_pretrained("nvidia/canary-1b-v2").eval()
ORIG  = copy.deepcopy(canary.cfg.decoding)
STRAT = ORIG.strategy
print("strategy:", STRAT)

def load(clip):
    a, _ = librosa.load(str(CLIPS / clip), sr=SR, mono=True)
    return a.astype(np.float32)

def _cap(p):
    return p if any(c.isupper() for c in p) else p[:1].upper() + p[1:]

def configure(beam, alpha, phrases):
    cfg = copy.deepcopy(ORIG)
    u = lambda k, v: OmegaConf.update(cfg, f"{STRAT}.{k}", v, force_add=True)
    u("beam_size", beam)
    if alpha > 0:
        u("boosting_tree.key_phrases_list", [_cap(p) for p in phrases])
        u("boosting_tree.source_lang",   "de")
        u("boosting_tree.context_score", 1.0)
        u("boosting_tree.depth_scaling", 1.0)
        u("boosting_tree_alpha", float(alpha))
    else:
        u("boosting_tree.key_phrases_list", None)
        u("boosting_tree_alpha", 0.0)
    canary.change_decoding_strategy(decoding_cfg=cfg)

def say(a):
    with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as f:
        sf.write(f.name, a, SR); p = f.name
    try:
        o = canary.transcribe([p], batch_size=1, verbose=False,
                              source_lang="de", target_lang="de", task="asr", pnc="yes")[0]
    finally:
        Path(p).unlink(missing_ok=True)
    return (o.text if hasattr(o, "text") else str(o)).strip()

ARMS = {k: float(v) for k, v in MAN["config"]["arms"].items()}
print("arms:", ARMS, "| helpers ready")

In [ ]:
# 3. Smoke test: up to three gold clips of one meeting, both arms; expect German text.
test = [s for s in SEGS if s["gold"]][:3]
for tag, alpha in ARMS.items():
    configure(4, alpha, TERMS.get(test[0]["meeting_id"], []))
    for s in test:
        if s["meeting_id"] != test[0]["meeting_id"]:
            continue
        print(f"{tag:15} {s['segment_id']}: {say(load(s['clip']))[:110]}")
canary.change_decoding_strategy(decoding_cfg=copy.deepcopy(ORIG))

In [ ]:
# 4. Full run, resumable: gold segments first, then the rest; saves every 50 decodes.
# Both arms of a meeting run before the next one, so finished meetings are always paired.
P = OUT / "sweep_full.json"
sweep = json.loads(P.read_text(encoding="utf-8")) if P.exists() else {}
for tag in ARMS: sweep.setdefault(tag, {})
t0, n, since_save = time.time(), 0, 0
for phase in (True, False):
    by_mid = {}
    for s in SEGS:
        if s["gold"] == phase:
            by_mid.setdefault(s["meeting_id"], []).append(s)
    for mid in sorted(by_mid):
        for tag, alpha in ARMS.items():
            todo = [s for s in by_mid[mid] if "text" not in sweep[tag].get(s["segment_id"], {})]
            if not todo:
                continue
            configure(4, alpha, TERMS.get(mid, []))
            for s in todo:
                try:
                    sweep[tag][s["segment_id"]] = {"meeting_id": mid, "text": say(load(s["clip"]))}
                except Exception as e:
                    sweep[tag][s["segment_id"]] = {"meeting_id": mid, "error": f"{type(e).__name__}: {e}"[:150]}
                n += 1; since_save += 1
                if since_save >= 50:
                    P.write_text(json.dumps(sweep, ensure_ascii=False), encoding="utf-8"); since_save = 0
            print(f"  {'gold' if phase else 'rest'} {mid:8} {tag:15} done  ({n} decodes, {time.time()-t0:.0f}s)", flush=True)
P.write_text(json.dumps(sweep, ensure_ascii=False), encoding="utf-8")
canary.change_decoding_strategy(decoding_cfg=copy.deepcopy(ORIG))
for tag in ARMS:
    ok = sum("text" in v for v in sweep[tag].values())
    print(f"{tag}: {ok}/{len(SEGS)} decoded, {len(sweep[tag]) - ok} errors")

In [ ]:
# 5. Sanity check, then zip bias_full/ into bias_full_results.zip.
a, b = (sweep[t] for t in ARMS)
diff = sum(1 for k in a if "text" in a[k] and "text" in b.get(k, {}) and a[k]["text"] != b[k]["text"])
print(f"segments where boosting changed the text: {diff} of {len(a)}")
print("If this is 0, boosting was inert: stop and report, do not score.")
import shutil
shutil.make_archive("bias_full_results", "zip", ".", "bias_full")
print("download bias_full_results.zip")